# Week 2: CNNs & RNNs - Transfer Learning for Computer Vision

**Bread Financial - AI for Data Scientists Academy**

---

## Learning Objectives

By the end of this session, you will be able to:

- Understand how Convolutional Neural Networks (CNNs) process images
- Explain convolution and pooling operations
- Load and use pretrained models from torchvision
- Apply transfer learning to fine-tune models for custom tasks
- Prepare image datasets with proper transforms and augmentation
- Train a state-of-the-art image classifier using transfer learning
- Achieve >85% accuracy on image classification tasks

## Prerequisites

Before starting this notebook, you should have:

- Completed Week 1: PyTorch Basics (tensors, autograd, nn.Module, training loops)
- Watched pre-class videos on: CNNs (convolution, pooling, architectures), RNNs (sequence modeling, hidden states)
- Basic understanding of neural networks and backpropagation

## Session Format

- **2-hour hands-on session**
- Instructor will demo key concepts (live coding)
- You will complete labs independently
- Solutions shared after class

---

## Important: GPU Setup for Google Colab

**GPU is STRONGLY RECOMMENDED for this week's labs** (CNNs train much slower on CPU).

If you're running this notebook on Google Colab:

1. Click on **Runtime** in the top menu
2. Select **Change runtime type**
3. Under **Hardware accelerator**, select **T4 GPU**
4. Click **Save**

You can verify GPU access by running the import cell below - it will show your GPU name.

---

## Section 0: Environment Setup

Let's start by setting up our environment and understanding what we're building today.

In [ ]:
# Install required packages (run this first in Google Colab)
# If running locally with conda/venv, you may skip this cell

!pip install torch torchvision matplotlib pillow scikit-learn

In [ ]:
# Import all necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets, models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import copy
import time

# Check PyTorch and torchvision versions
print(f"PyTorch version: {torch.__version__}")
print(f"torchvision version: {torchvision.__version__}")

# Device configuration - automatically use GPU if available
# GPU is CRITICAL for CNNs - they train much faster than on CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\nGPU is available - training will be fast!")
else:
    print("\nGPU not available - training will be slower.")
    print("On Colab: Runtime -> Change runtime type -> T4 GPU")

# Set random seed for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("\n✅ Environment setup complete!")

## What Are We Building Today?

### Real-World Context: Automated Document Classification for Fraud Detection

Imagine you're a data scientist at Bread Financial's fraud detection team. When customers apply for credit cards, they must submit identity verification documents:

- Passports
- Driver's licenses
- Utility bills
- Bank statements
- Social security cards

**Current problem**: Clerks manually review and classify thousands of document images daily. This is:
- Slow (processing bottleneck)
- Expensive (labor costs)
- Error-prone (misclassifications lead to fraud)
- Not scalable (application volume growing)

**Your mission**: Build an AI system that automatically classifies document types from images.

### The Challenge: Limited Data

Here's the catch - you have only a few thousand labeled document images (proprietary data is scarce). Building a CNN from scratch requires millions of images.

**Solution: Transfer Learning**

Instead of training from scratch, you'll use a model **pre-trained on ImageNet** (14 million images, 1000 categories). This model already learned to recognize edges, shapes, textures, and objects. You'll **fine-tune** it for your specific task.

This is how **real-world computer vision systems are built** - transfer learning is the industry standard!

### Today's Dataset

For this class, we'll use **CIFAR-10** (public dataset of 60,000 images in 10 categories: planes, cars, birds, etc.). The transfer learning process is **identical** to what you'd do with proprietary document images.

Let's preview what we're working with:

In [ ]:
# Load CIFAR-10 dataset to preview
# CIFAR-10: 60,000 32x32 color images in 10 classes
transform_preview = transforms.Compose([transforms.ToTensor()])

# Download and load training data
cifar_preview = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_preview)

# Class names
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# Display 20 sample images (2 rows x 10 columns)
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flat):
    image, label = cifar_preview[i]
    # Convert tensor to numpy for display: (C, H, W) -> (H, W, C)
    image_np = image.permute(1, 2, 0).numpy()
    ax.imshow(image_np)
    ax.set_title(classes[label], fontsize=9)
    ax.axis('off')

plt.suptitle('CIFAR-10 Dataset - Sample Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nDataset size: {len(cifar_preview):,} training images")
print(f"Image dimensions: 32x32 pixels (RGB color)")
print(f"Number of classes: {len(classes)}")
print(f"\n🎯 Goal: Use transfer learning to achieve >85% accuracy!")

---

# Topic 1: Understanding CNNs - Convolution and Pooling

## What is Convolution?

**Convolution** slides a small filter (3×3 or 5×5) over an image, detecting patterns like edges, textures, or shapes.

Think of it as a pattern detector that scans the entire image looking for specific features.

Let's see it in action with edge detection:

In [ ]:
# Example: Edge Detection with Convolution

# Get a sample image from CIFAR-10
sample_image, _ = cifar_preview[100]

# Convert to grayscale for easier visualization
gray_image = 0.299 * sample_image[0] + 0.587 * sample_image[1] + 0.114 * sample_image[2]
gray_image = gray_image.unsqueeze(0).unsqueeze(0)  # Add batch and channel dims

# Define edge detection filter (Sobel filter for horizontal edges)
horizontal_filter = torch.tensor([
    [[-1., -2., -1.],
     [ 0.,  0.,  0.],
     [ 1.,  2.,  1.]]
])

# Apply convolution - this is what nn.Conv2d does!
horizontal_edges = torch.nn.functional.conv2d(
    gray_image, 
    horizontal_filter.unsqueeze(0),
    padding=1
)

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(gray_image.squeeze(), cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(horizontal_edges.squeeze(), cmap='gray')
axes[1].set_title('After Convolution\\n(Horizontal edges detected)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("💡 The convolution filter detected horizontal edges automatically!")
print("   CNNs learn these filters during training.")

## What is Pooling?

**Pooling** reduces image size by summarizing regions. **Max pooling** takes the maximum value in each 2×2 region.

Why? Reduces computation and makes the network less sensitive to small shifts in the image.

Let's see how it reduces image size:

In [ ]:
# Example: Max Pooling

# Apply 2x2 max pooling to the edge-detected image
pooled = torch.nn.functional.max_pool2d(horizontal_edges, kernel_size=2, stride=2)

# Visualize before and after pooling
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(horizontal_edges.squeeze(), cmap='gray')
axes[0].set_title(f'Before Pooling: {horizontal_edges.shape[2]}×{horizontal_edges.shape[3]}')
axes[0].axis('off')

axes[1].imshow(pooled.squeeze(), cmap='gray')
axes[1].set_title(f'After Pooling: {pooled.shape[2]}×{pooled.shape[3]}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"💡 Pooling reduced image size by 75% ({horizontal_edges.shape[2]}×{horizontal_edges.shape[3]} → {pooled.shape[2]}×{pooled.shape[3]})")
print("   Important features are preserved, computation is reduced!")

## Building a Simple CNN

Now let's combine convolution and pooling to build a complete CNN for CIFAR-10.

Architecture: **Conv → ReLU → Pool → Conv → ReLU → Pool → Flatten → Fully Connected**

In [ ]:
# Building a Simple CNN from Scratch

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # First conv block: 3 input channels (RGB) → 16 feature maps
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)  # Reduces size by half
        
        # Second conv block: 16 → 32 feature maps
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        
        # Fully connected layers
        # After 2 pooling layers: 32×32 → 16×16 → 8×8
        # So: 32 channels × 8 × 8 = 2048 features
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)  # 10 classes
    
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))  # Conv → ReLU → Pool
        x = self.pool(torch.relu(self.conv2(x)))  # Conv → ReLU → Pool
        x = x.view(x.size(0), -1)  # Flatten
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Create model
simple_cnn = SimpleCNN().to(device)
print(simple_cnn)

# Test with dummy data
dummy_input = torch.randn(4, 3, 32, 32).to(device)
output = simple_cnn(dummy_input)
print(f"\\nInput: {dummy_input.shape} → Output: {output.shape}")
print(f"Total parameters: {sum(p.numel() for p in simple_cnn.parameters()):,}")

---

## Lab 1: Build and Train Your Own CNN

Now it's your turn! Build a deeper CNN and train it on CIFAR-10.

**Task**: Create a CNN with this architecture:
- Conv(3→32) → ReLU → Pool
- Conv(32→64) → ReLU → Pool  
- Conv(64→128) → ReLU → Pool
- Flatten → FC(2048→256) → ReLU → FC(256→10)

Then train it for 5 epochs and see how well it performs!

In [ ]:
# Lab 1: Build Your Own CNN

class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR CODE: Define 3 conv blocks and 2 FC layers
        # Conv1: 3 → 32
        self.conv1 = None  # YOUR CODE
        
        # Conv2: 32 → 64
        self.conv2 = None  # YOUR CODE
        
        # Conv3: 64 → 128
        self.conv3 = None  # YOUR CODE
        
        # Pooling layer (reuse for all 3 blocks)
        self.pool = None  # YOUR CODE
        
        # FC layers
        # After 3 pooling: 32×32 → 16×16 → 8×8 → 4×4
        # So: 128 × 4 × 4 = 2048
        self.fc1 = None  # YOUR CODE: 2048 → 256
        self.fc2 = None  # YOUR CODE: 256 → 10
    
    def forward(self, x):
        # YOUR CODE: Implement forward pass
        # Pattern: conv → relu → pool (3 times), then flatten, then fc1 → relu → fc2
        pass

# Test your model (uncomment when ready)
# my_cnn = MyCNN().to(device)
# test_input = torch.randn(2, 3, 32, 32).to(device)
# test_output = my_cnn(test_input)
# print(f\"Input: {test_input.shape} → Output: {test_output.shape}\")  # Should be (2, 10)

---

# Topic 2: Transfer Learning - Use Pretrained Models

## Why Transfer Learning?

Building a CNN from scratch requires:
- Millions of images
- Days/weeks of training  
- Expensive GPUs

**Transfer learning** lets us use models already trained on ImageNet (14M images, 1000 classes). These models learned to detect edges, textures, shapes, and objects.

We just replace the final layer and fine-tune for our task!

Let's load a pretrained model:

In [ ]:
# Load a pretrained ResNet18 model

# Load model pretrained on ImageNet
resnet18 = models.resnet18(pretrained=True)
print(resnet18)

# Look at the final layer
print(f"\nOriginal final layer: {resnet18.fc}")
print(f"It outputs 1000 classes (ImageNet classes)")
print(f"\nWe need to replace this with a layer that outputs 10 classes (CIFAR-10)!")

## Replacing the Final Layer

To adapt the pretrained model for CIFAR-10 (10 classes), we replace the final fully connected layer:

```python
# Original: Linear(512, 1000) for ImageNet
# New: Linear(512, 10) for CIFAR-10
model.fc = nn.Linear(512, 10)
```

Let's do it:

In [ ]:
# Replace the final layer for CIFAR-10

# Get the number of input features to the final layer
num_features = resnet18.fc.in_features
print(f"Final layer input features: {num_features}")

# Replace the final layer: 512 → 10 classes
resnet18.fc = nn.Linear(num_features, 10)
print(f"\nNew final layer: {resnet18.fc}")

# Move to device
resnet18 = resnet18.to(device)

# Test with dummy CIFAR-10 input
test_input = torch.randn(2, 3, 32, 32).to(device)
test_output = resnet18(test_input)
print(f"\nInput: {test_input.shape} → Output: {test_output.shape}")
print("✅ Model now outputs 10 classes for CIFAR-10!")

---

# Topic 3: Preparing Data for Transfer Learning

## Image Transforms and Data Augmentation

CNNs need properly sized and normalized images. We also use **data augmentation** to artificially expand our dataset:
- Random flips
- Random crops  
- Color jittering

This helps prevent overfitting!

Let's set up transforms for CIFAR-10:

In [ ]:
# Data Transforms for Training and Validation

# Training transforms: augmentation + normalization
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # Randomly flip images horizontally
    transforms.RandomCrop(32, padding=4),  # Random crop with padding
    transforms.ToTensor(),  # Convert PIL image to tensor
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))  # CIFAR-10 mean/std
])

# Validation/test transforms: only normalization (no augmentation!)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

print("✅ Transforms ready!")
print("\\nTraining: augmentation + normalization")
print("Testing: normalization only (no random changes)")

## Loading Data with DataLoaders

Now let's load CIFAR-10 with proper transforms and create train/validation/test splits:

In [ ]:
# Load CIFAR-10 datasets

# Training data (with augmentation)
train_dataset_full = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)

# Split into train (45k) and validation (5k)
train_size = 45000
val_size = 5000
train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

# Test data (10k images, no augmentation)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f\"Train: {len(train_dataset):,} images ({len(train_loader)} batches)")
print(f\"Validation: {len(val_dataset):,} images ({len(val_loader)} batches)")
print(f\"Test: {len(test_dataset):,} images ({len(test_loader)} batches)")
print(f\"\\nBatch size: {batch_size}\")

---

# Topic 4: The Main Lab - Transfer Learning on CIFAR-10

## Complete Transfer Learning Workflow

Now let's put everything together! We'll:
1. Load pretrained ResNet18
2. Replace final layer for CIFAR-10
3. Train with a proper training loop
4. Validate on validation set
5. Achieve >85% accuracy!

Here's the complete training function:

In [ ]:
# Training function with validation

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    """
    Train the model and track training/validation accuracy
    """
    train_losses = []
    val_accuracies = []
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Training step: zero → forward → loss → backward → step
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        avg_train_loss = running_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_acc = 100 * correct / total
        val_accuracies.append(val_acc)
        
        print(f'Epoch [{epoch+1}/{num_epochs}] Loss: {avg_train_loss:.4f}, Val Acc: {val_acc:.2f}%')
    
    return train_losses, val_accuracies

print(\"✅ Training function ready!\")

## Lab 2: MAIN LAB - Fine-Tune ResNet18 on CIFAR-10

This is the main outcome of today's class! You'll:
1. Create a transfer learning model
2. Set up loss function and optimizer
3. Train for 10 epochs
4. Achieve >85% accuracy

**Your turn!**

In [ ]:
# Lab 2: Transfer Learning - Complete Pipeline

# Step 1: Create model
# YOUR CODE: Load pretrained ResNet18 and replace final layer
model = None  # YOUR CODE: models.resnet18(pretrained=True)
# YOUR CODE: Replace model.fc with nn.Linear(?, 10)
# YOUR CODE: Move model to device

# Step 2: Loss and optimizer
criterion = None  # YOUR CODE: nn.CrossEntropyLoss()
optimizer = None  # YOUR CODE: optim.Adam(model.parameters(), lr=0.001)

# Step 3: Train!
# YOUR CODE: Call train_model() function
# train_losses, val_accs = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10)

# Step 4: Plot results
# YOUR CODE: Plot training loss and validation accuracy over epochs
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].plot(train_losses)
# axes[0].set_title('Training Loss')
# axes[0].set_xlabel('Epoch')
# axes[0].set_ylabel('Loss')
# 
# axes[1].plot(val_accs)
# axes[1].set_title('Validation Accuracy')
# axes[1].set_xlabel('Epoch')
# axes[1].set_ylabel('Accuracy (%)')
# plt.tight_layout()
# plt.show()

print(\"Complete the code above to train your transfer learning model!\")

---

# Topic 5: Evaluation and Analysis

## Test Your Model

After training, let's evaluate on the test set and see how well transfer learning worked!

In [ ]:
# Evaluate on test set

def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = 100 * correct / total
    print(f'Test Accuracy: {accuracy:.2f}%')
    
    if accuracy >= 85:
        print(\"\\n🎉 Congratulations! You achieved >85% with transfer learning!\")
    else:
        print(\"\\n💡 Try training for more epochs or adjusting the learning rate.\")
    
    return accuracy, all_preds, all_labels

# Uncomment to evaluate your trained model
# test_acc, preds, labels = evaluate_model(model, test_loader)

## Visualize Predictions

Let's see what our model got right and wrong:

In [ ]:
# Visualize predictions

# Uncomment to visualize (after training your model)
# 
# # Get one batch from test set
# images_batch, labels_batch = next(iter(test_loader))
# images_batch, labels_batch = images_batch.to(device), labels_batch.to(device)
# 
# # Get predictions
# model.eval()
# with torch.no_grad():
#     outputs = model(images_batch)
#     _, predictions = torch.max(outputs, 1)
# 
# # Move back to CPU for visualization
# images_batch = images_batch.cpu()
# predictions = predictions.cpu()
# labels_batch = labels_batch.cpu()
# 
# # Plot 10 examples
# fig, axes = plt.subplots(2, 5, figsize=(12, 5))
# for i, ax in enumerate(axes.flat):
#     # Denormalize image for display
#     img = images_batch[i]
#     img = img * torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1) + torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
#     img = img.permute(1, 2, 0).numpy()
#     img = np.clip(img, 0, 1)
#     
#     ax.imshow(img)
#     pred_class = classes[predictions[i]]
#     true_class = classes[labels_batch[i]]
#     color = 'green' if predictions[i] == labels_batch[i] else 'red'
#     ax.set_title(f'Pred: {pred_class}\\nTrue: {true_class}', color=color)
#     ax.axis('off')
# 
# plt.tight_layout()
# plt.show()

print(\"Uncomment the code above to visualize predictions after training!\")

---

# Optional Labs (Complete at Home)

## Optional Lab: Compare Different Pretrained Models

Try other pretrained models and compare performance:
- ResNet34, ResNet50 (deeper ResNets)
- VGG16 (different architecture)
- MobileNetV2 (lightweight, faster)

Which performs best? Which trains fastest?

In [ ]:
# Optional Lab: Try Different Models

# Example: Try VGG16
# vgg16 = models.vgg16(pretrained=True)
# num_features = vgg16.classifier[6].in_features
# vgg16.classifier[6] = nn.Linear(num_features, 10)
# vgg16 = vgg16.to(device)
#
# # Train and compare!

# Example: Try MobileNetV2  
# mobilenet = models.mobilenet_v2(pretrained=True)
# num_features = mobilenet.classifier[1].in_features
# mobilenet.classifier[1] = nn.Linear(num_features, 10)
# mobilenet = mobilenet.to(device)
#
# # Train and compare!

print(\"Try different pretrained models and compare their performance!\")

---

# Congratulations!

You've completed Week 2: CNNs & Transfer Learning!

## What You've Learned

✅ How convolution and pooling work in CNNs  
✅ Building CNNs from scratch with PyTorch  
✅ Loading pretrained models (ResNet, VGG, etc.)  
✅ Transfer learning: replacing final layers for custom tasks  
✅ Data augmentation and transforms  
✅ Complete training pipeline with validation  
✅ Achieving >85% accuracy with transfer learning  

## Key Takeaways

1. **Transfer learning is the industry standard** - don't train from scratch!
2. **Data augmentation** helps prevent overfitting on small datasets
3. **Always validate** during training to catch overfitting early
4. **Pretrained models** save time and achieve better results

## Next Steps

- **Week 3**: Spark ML on Databricks - distributed machine learning
- **Practice**: Try transfer learning on other datasets (Stanford Dogs, Food-101)
- **Challenge**: Can you achieve >90% accuracy on CIFAR-10?

## Resources

- [PyTorch Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [torchvision.models Documentation](https://pytorch.org/vision/stable/models.html)
- [CIFAR-10 Dataset](https://www.cs.toronto.edu/~kriz/cifar.html)

---

**Great work! See you next week for Spark ML on Databricks!**